# Tiny ears: an 8–10 MB bilingual sound recognizer (tiny-speech)

Trains a small model (about 7.5M parameters, **about 8.9 MB at 8 bits**) to hear the same Spanish and English sounds as a
large one (wav2vec2 espeak, 300M), on free Mexican and Latin American Spanish speech and some English. Code and details:
`tiny-speech/training/ears/` in github.com/larsboesen-prog/ranked-feed (branch `claude/tiny-speech-bilingual-v7lyp0`).

**Kaggle** (recommended: 30 free GPU hours a week, runs with the browser closed): Settings → Accelerator **GPU T4**,
Internet **on** (both need a phone-verified account), then **Save Version → Save & Run All**. Results: the notebook's
Output tab (`tiny-ears/onnx/ears-int8.onnx`).
**Colab**: Runtime → Change runtime type → **T4 GPU**, then Run all; results go to Google Drive (`MyDrive/tiny-ears`), and
running again after a disconnect resumes.

One session (about 11 hours): download and convert ~100 hours of speech, the teacher labels it, the student trains with
the time left, then the 8-bit ONNX file is exported and checked (at most 10 MB; same numbers as PyTorch).

In [ ]:
import os, sys, shutil, pathlib, json, subprocess
KAGGLE = os.path.exists("/kaggle/working")
try:
    import google.colab  # noqa: F401
    COLAB = True
except ImportError:
    COLAB = False

if COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = pathlib.Path("/content/drive/MyDrive/tiny-ears")   # kept between sessions: a new session resumes
    BASE = pathlib.Path("/content")
    HOURS = 11.0
elif KAGGLE:
    OUT = pathlib.Path("/kaggle/working/tiny-ears")           # the notebook's Output (keep it small: checkpoints, ONNX, logs)
    places = []
    for p in ("/kaggle/tmp", "/tmp"):
        try:
            os.makedirs(p, exist_ok=True)
            places.append(pathlib.Path(p))
        except OSError:
            pass
    BASE = max(places, key=lambda p: shutil.disk_usage(p).free)  # the roomiest scratch disk
    HOURS = 11.0                                              # Kaggle stops a session at 12 hours
else:
    OUT, BASE, HOURS = pathlib.Path("tiny-ears").resolve(), pathlib.Path(".").resolve(), 11.0
WORK = BASE / "ears-work"                                     # audio and labels: large, thrown away at the end
REPO = BASE / "ranked-feed"
OUT.mkdir(parents=True, exist_ok=True)
print({"kaggle": KAGGLE, "colab": COLAB, "out": str(OUT), "work": str(WORK), "free_gb": round(shutil.disk_usage(BASE).free / 1e9, 1)})
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# the code (public repository)
if not REPO.exists():
    !git clone -q --depth 1 --branch claude/tiny-speech-bilingual-v7lyp0 https://github.com/larsboesen-prog/ranked-feed.git {REPO}
else:
    !git -C {REPO} pull -q
!git -C {REPO} log -1 --format="%h %s" | tee {OUT}/code-version.txt
EARS = REPO / "tiny-speech/training/ears"

In [ ]:
# everything, in order; each finished stage is skipped when run again (resume after a stop or in a new session)
!{sys.executable} {EARS}/run.py all --work {WORK} --out {OUT} --hours {HOURS}

In [ ]:
# what came out
report = OUT / "onnx/report.json"
if report.exists():
    print(report.read_text())
for p in sorted(OUT.rglob("*")):
    if p.is_file() and p.suffix in (".onnx", ".json", ".pt", ".jsonl"):
        print(f"{p.stat().st_size / 1e6:8.2f} MB  {p.relative_to(OUT)}")